# ML distribuito con Spark MLlib

## Costruire una pipeline ML

### 1. Scarica il dataset Adult UCI

In [3]:
import os
import sys
import shutil
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Su Windows: decommenta e adatta se winutils non e' gia' nelle variabili di sistema
# os.environ["HADOOP_HOME"] = r"C:\hadoop"
# os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

print("Interprete    :", sys.executable)
print("HADOOP_HOME   :", os.environ.get("HADOOP_HOME", "NON IMPOSTATO"))
print("SPARK_LOCAL_IP:", os.environ.get("SPARK_LOCAL_IP"))

Interprete    : C:\Users\Allysa\Documents\LAB - Machine Learning\.venv\Scripts\python.exe
HADOOP_HOME   : C:\Users\Allysa\Downloads\hadoop-3.5.0
SPARK_LOCAL_IP: 127.0.0.1


In [4]:
# Dimostrazione: le config di una seconda builder vengono ignorate
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MachineLearningPyspark").getOrCreate()

prova = SparkSession.builder.config("spark.sql.shuffle.partitions", "999").getOrCreate()
print("Valore effettivo dopo la seconda builder:",
      prova.conf.get("spark.sql.shuffle.partitions"), " <- non e' 999")

# Per riconfigurare davvero: prima si ferma
spark.stop()

spark = (
    SparkSession.builder
    .appName("WineQualityPreprocessing")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python.authenticate.socketTimeout", "60")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark  :", spark.version)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Valore effettivo dopo la seconda builder: 999  <- non e' 999
Spark  : 4.2.0
Shuffle: 4


In [5]:
# Scarichiamo il file adult.data dal repository UCI
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -O adult.csv


"wget" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


In [6]:
import urllib.request
import urllib.error

URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data")

DATA_DIR = Path(r"C:\sparklab\data") if os.name == "nt" else Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "adult.csv"

if CSV_PATH.exists():
    print("Gia' presente in cache:", CSV_PATH)
else:
    try:
        urllib.request.urlretrieve(URL, CSV_PATH)
        print("Scaricato in:", CSV_PATH)
    except urllib.error.URLError as e:
        raise SystemExit(
            f"Download fallito ({e.reason}). L'URL di UCI cambia periodicamente: "
            f"scaricate il file a mano e mettetelo in {CSV_PATH}"
        )

print("Dimensione:", CSV_PATH.stat().st_size, "byte")
print(CSV_PATH.read_text(encoding="utf-8").splitlines()[0])

Gia' presente in cache: C:\sparklab\data\adult.csv
Dimensione: 3974305 byte
39, State-gov, 77516, Bachelors, 13, Never-married, Adm-clerical, Not-in-family, White, Male, 2174, 0, 40, United-States, <=50K


### 2. Installa PySpark e avvia la sessione

In [7]:
# Inizializza SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("AdultIncomeML").getOrCreate()


### 3. Carica e prepara i dati

In [8]:
# Definizione delle colonne (da documentazione UCI)
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]

# Leggiamo il CSV e assegniamo i nomi di colonna
df = spark.read.csv("adult.csv", header=False, inferSchema=True).toDF(*columns)

# Rimuoviamo eventuali righe con valori nulli
df = df.dropna()

# Convertiamo la colonna 'income' in un'etichetta numerica: >50K = 1, altrimenti 0
from pyspark.sql.functions import when
df = df.withColumn("label", when(df.income == " >50K", 1).otherwise(0))

df.show(10)

+---+-----------------+--------+----------+-------------+--------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------+-----+
|age|        workclass|  fnlwgt| education|education_num|      marital_status|        occupation|  relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|native_country|income|label|
+---+-----------------+--------+----------+-------------+--------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------+-----+
| 39|        State-gov| 77516.0| Bachelors|         13.0|       Never-married|      Adm-clerical| Not-in-family| White|   Male|      2174.0|         0.0|          40.0| United-States| <=50K|    0|
| 50| Self-emp-not-inc| 83311.0| Bachelors|         13.0|  Married-civ-spouse|   Exec-managerial|       Husband| White|   Male|         0.0|         0.0|          13.0| United-States| <=50K|    0|
| 38|          

### 4. Costruzione della pipeline ML

In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Colonne categoriche da trasformare
cat_cols = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country"
]

# Colonne numeriche che useremo direttamente
num_cols = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]

# Per ogni colonna categorica, creiamo:
# - uno StringIndexer (per codifica numerica)
# - un OneHotEncoder (per codifica binaria vettoriale)
# Set handleInvalid to 'keep' to handle unseen labels in the test set
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid='keep') for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec") for c in cat_cols]

# Assembler finale: unisce tutte le feature numeriche e codificate in un vettore unico
assembler = VectorAssembler(
    inputCols=[c + "_vec" for c in cat_cols] + num_cols,
    outputCol="features"
)

# Modello finale: regressione logistica
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Costruzione della pipeline completa
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])


[StringIndexer_204c64e0bebf, StringIndexer_7d4f56d3df44, StringIndexer_b80f0f142aad, StringIndexer_591aed7113ea, StringIndexer_9eed445916c8, StringIndexer_aa5f12a79205, StringIndexer_9ad2deb3fa3e, StringIndexer_30200473e40c]


### 6. Addestramento del modello

In [10]:
# Suddividiamo il dataset in training e test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Alleniamo la pipeline sul training set
pipeline_model = pipeline.fit(train_df)

# Usiamo il modello per fare predizioni sul test set
predictions = pipeline_model.transform(test_df)

# Visualizziamo alcune predizioni
predictions.select("label", "prediction", "probability").show(5, truncate=False)


+-----+----------+------------------------------------------+
|label|prediction|probability                               |
+-----+----------+------------------------------------------+
|0    |0.0       |[0.9996847756378485,3.152243621514783E-4] |
|0    |0.0       |[0.9990853441638308,9.146558361692314E-4] |
|0    |0.0       |[0.9996323976371279,3.6760236287214365E-4]|
|0    |0.0       |[0.9992164563340913,7.835436659087147E-4] |
|0    |0.0       |[0.999745538193205,2.544618067950033E-4]  |
+-----+----------+------------------------------------------+
only showing top 5 rows


### 6. Valutazione con AUC

In [11]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Usiamo l'AUC come metrica di valutazione
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"AUC: {auc:.4f}")


AUC: 0.9064


### 7. Confusion Matrix (semplificata)

In [12]:
# Crosstab per ottenere la matrice di confusione (label vs prediction)
predictions.crosstab("label", "prediction").show()


+----------------+----+---+
|label_prediction| 0.0|1.0|
+----------------+----+---+
|               1| 629|948|
|               0|4568|340|
+----------------+----+---+



### 8. Salvataggio del modello addestrato

In [16]:
# Salva il modello serializzato su disco (ad esempio, su Google Drive)
pipeline_model.save("adult_income_pipeline_model")
#Fallisce perché la cartella esisteva già.
#Per sovrascrivere prova: pipeline_model.write().overwrite().save("adult_income_pipeline_model")


Py4JJavaError: An error occurred while calling o2206.save.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: Hadoop home directory C:\Users\Allysa\Downloads\hadoop-3.5.0 does not exist -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.had

### 9. Sostituire Logistic Regression con Random Forest

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Creiamo un nuovo modello Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label")

# Ricostruiamo la pipeline con lo stesso preprocessing + Random Forest
pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])


### 10. Addestramento con Random Forest

In [ ]:
# Alleniamo la pipeline Random Forest
pipeline_rf_model = pipeline_rf.fit(train_df)

# Predizioni sul test set
predictions_rf = pipeline_rf_model.transform(test_df)

# Visualizziamo le prime predizioni
predictions_rf.select("label", "prediction", "probability").show(5, truncate=False)


### 11. Valutazione di Random Forest (AUC)

In [ ]:
# Valutiamo il modello Random Forest con AUC
evaluator_rf = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc_rf = evaluator_rf.evaluate(predictions_rf)
print(f"AUC con Random Forest: {auc_rf:.4f}")


### 12. Tuning iperparametri con CrossValidator

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Usiamo sempre Random Forest come classificatore
rf_tuned = RandomForestClassifier(featuresCol="features", labelCol="label")

# Ricostruiamo la pipeline
pipeline_tuned = Pipeline(stages=indexers + encoders + [assembler, rf_tuned])

# Costruiamo la griglia di parametri da testare
paramGrid = ParamGridBuilder() \
    .addGrid(rf_tuned.numTrees, [2, 3]) \
    .addGrid(rf_tuned.maxDepth, [1, 2]) \
    .build()


In [ ]:
# Definiamo il validatore incrociato (3-fold)
crossval = CrossValidator(
    estimator=pipeline_tuned,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
    numFolds=3,
    parallelism=2  # se vuoi velocizzare in ambienti multi-core
)


In [ ]:
# Allenamento con validazione incrociata
cv_model = crossval.fit(train_df)

# Predizioni sul test set
cv_predictions = cv_model.transform(test_df)

# Valutazione finale
auc_cv = evaluator_rf.evaluate(cv_predictions)
print(f"AUC dopo tuning iperparametri: {auc_cv:.4f}")


### 13. Visualizzare l'importanza delle feature (Random Forest)

In [ ]:
# Estraiamo il modello Random Forest allenato dal pipeline_model (dopo tuning)
rf_model = cv_model.bestModel.stages[-1]  # Ultimo stage della pipeline è RandomForestClassifier

# Otteniamo i nomi delle feature effettive usate nell'assembler
input_features = [c + "_vec" for c in cat_cols] + num_cols

# L'assembler ha già creato il vettore finale → possiamo accedere all'importanza delle feature
importances = rf_model.featureImportances


In [ ]:
# Per visualizzare: trasformiamo in Pandas DataFrame (semplificato)
import pandas as pd
import matplotlib.pyplot as plt

# Convertiamo SparseVector in lista
importance_list = importances.toArray().tolist()

# Poiché le one-hot encoder producono molte colonne, possiamo usare solo i primi N
N = 20  # Numero di feature più importanti da visualizzare

# Creiamo DataFrame ordinato
imp_df = pd.DataFrame({
    "feature_index": list(range(len(importance_list))),
    "importance": importance_list
}).sort_values(by="importance", ascending=False).head(N)

# Visualizza
plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature_index"], imp_df["importance"])
plt.xlabel("Importanza")
plt.ylabel("Indice Feature")
plt.title("Top 20 Feature Importance - Random Forest")
plt.gca().invert_yaxis()
plt.show()


###  14. Tracciare il grafico ROC Curve

Spark ML non fornisce direttamente le curve ROC come sklearn, ma puoi farlo manualmente:

In [ ]:
from sklearn.metrics import roc_curve, auc

# Estrai probabilità e label dal DataFrame Spark
probs_labels = cv_predictions.select("probability", "label").rdd.map(
    lambda row: (float(row.probability[1]), row.label)
).collect()

# Dividi in array
y_scores, y_true = zip(*probs_labels)

# Calcola ROC con sklearn
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.grid(True)
plt.show()
